# Extract selected IBTrACS columns

This notebook reads `input/ibtracs.ALL.list.v04r01.csv` and writes a new CSV containing only the requested columns. It preserves the original two-line IBTrACS header style: the first row contains column names and the second row contains units.

In [ ]:
from pathlib import Path
import csv

input_csv = Path("input/ibtracs.ALL.list.v04r01.csv")
output_csv = Path("input/ibtracs.ALL.list.v04r01.selected_columns.csv")

columns_to_keep = [
    "SID",
    "SEASON",
    "NUMBER",
    "BASIN",
    "SUBBASIN",
    "NAME",
    "ISO_TIME",
    "NATURE",
    "LAT",
    "LON",
    "WMO_WIND",
    "WMO_PRES",
]

input_csv, output_csv

In [ ]:
with input_csv.open("r", newline="") as src:
    reader = csv.reader(src)
    header = next(reader)
    units = next(reader)

missing_columns = [column for column in columns_to_keep if column not in header]
if missing_columns:
    raise ValueError(f"Missing columns in {input_csv}: {missing_columns}")

column_indices = [header.index(column) for column in columns_to_keep]
wmo_pres_index = header.index("WMO_PRES")

with input_csv.open("r", newline="") as src, output_csv.open("w", newline="") as dst:
    reader = csv.reader(src)
    writer = csv.writer(dst, lineterminator="\n")

    header = next(reader)
    units = next(reader)

    writer.writerow([header[index] for index in column_indices])
    writer.writerow([units[index] for index in column_indices])

    rows_written = 0
    rows_skipped = 0

    for row in reader:
        if not row[wmo_pres_index].strip():
            rows_skipped += 1
            continue
        writer.writerow([row[index] for index in column_indices])
        rows_written += 1

print(f"Wrote {output_csv}")
print(f"Rows written: {rows_written}")
print(f"Rows skipped because WMO_PRES is missing: {rows_skipped}")

In [ ]:
with output_csv.open("r", newline="") as dst:
    for line_number, line in zip(range(5), dst):
        print(line.rstrip("\n"))